In [ ]:
import numpy as np
import numba
from numba import cuda
from time import time

44.9 ms ± 6.96 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:

@cuda.jit
def matmul(A, B, C):
    """Perform square matrix multiplication of C = A * B."""
    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        tmp = 0.
        for k in range(A.shape[1]):
            tmp += A[i, k] * B[k, j] # GPU线程每个独立运算，结果相互独立
        C[i, j] = tmp

In [ ]:
N = 1000

x_h = np.arange(N*N).reshape([N, N]).astype(np.float32)
y_h = np.ones([N, N]).astype(np.float32)
z_h = np.zeros([N, N]).astype(np.float32)

x_d = cuda.to_device(x_h)
y_d = cuda.to_device(y_h)
z_d = cuda.to_device(z_h)

threadsperblock = (16, 16)
blockspergrid_x = math.ceil(z_h.shape[0] / threadsperblock[0])
blockspergrid_y = math.ceil(z_h.shape[1] / threadsperblock[1])
blockspergrid = (blockspergrid_x, blockspergrid_y)

matmul[blockspergrid, threadsperblock](x_d, y_d, z_d)
z_h = z_d.copy_to_host()
print(z_h)
print(x_h @ y_h)

CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:

In [ ]:
%timeit x_h @ y_h

44.7 ms ± 8.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%timeit
matmul[blockspergrid, threadsperblock](x_d, y_d, z_d)
cuda.synchronize()

18.8 ms ± 261 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
TPB = 16 # 可以理解成threads per block的一个维度

@cuda.jit
def fast_matmul(A, B, C):
    # 创建2个16*16的share memory数组，放进share memory方便反复使用
    sA = cuda.shared.array(shape=(TPB, TPB), dtype=float32)
    sB = cuda.shared.array(shape=(TPB, TPB), dtype=float32)

    # 一个线程负责一个C[y, x]，y负责行x负责列
    x, y = cuda.grid(2)

    # 当前thread在自己block的位置，不是全局位置，tx ty的范围是是0到15
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    # grid在x方向上有多少个block
    bpg = cuda.gridDim.x # 63
    tmp = float32(0.)

    # 每次只能加载A和B的一个16×16tile要完成完整的点积，必须一块一块往后读
    for i in range(bpg):

        # 需要先清零因为最后一个tile不完整防止最后一个tile没有被填充的部分保留上次循环的内容
        sA[ty, tx] = 0
        sB[ty, tx] = 0
        # 固定y，一直读取A的第y行
        if y < A.shape[0] and (tx + i * TPB) < A.shape[1]: # (tx + i * TPB)决定读取到哪一列
            sA[ty, tx] = A[y, tx + i * TPB] # 固定y不断右移
        if x < B.shape[1] and (ty + i * TPB) < B.shape[0]:
            sB[ty, tx] = B[ty + i * TPB, x]

        # Wait until all threads finish preloading
        cuda.syncthreads()

        # Computes partial product on the shared memory
        for j in range(TPB):
            tmp += sA[ty, j] * sB[j, tx]

        # Wait until all threads finish computing
        cuda.syncthreads()

    if y < C.shape[0] and x < C.shape[1]:
        C[y, x] = tmp

In [ ]:
x_d = cuda.to_device(x_h)
y_d = cuda.to_device(y_h)
z_d = cuda.to_device(z_h)

threadsperblock = (TPB, TPB)
blockspergrid_x = math.ceil(z_h.shape[0] / threadsperblock[0])
blockspergrid_y = math.ceil(z_h.shape[1] / threadsperblock[1])
blockspergrid = (blockspergrid_x, blockspergrid_y)

fast_matmul[blockspergrid, threadsperblock](x_d, y_d, z_d)
z_h = z_d.copy_to_host()
print(z_h)
print(x_h @ y_h)

[[4.995000e+05 4.995000e+05 4.995000e+05 ... 4.995000e+05 4.995000e+05
  4.995000e+05]
 [1.499500e+06 1.499500e+06 1.499500e+06 ... 1.499500e+06 1.499500e+06
  1.499500e+06]
 [2.499500e+06 2.499500e+06 2.499500e+06 ... 2.499500e+06 2.499500e+06
  2.499500e+06]
 ...
 [9.974994e+08 9.974994e+08 9.974994e+08 ... 9.974994e+08 9.974994e+08
  9.974994e+08]
 [9.984989e+08 9.984989e+08 9.984989e+08 ... 9.984989e+08 9.984989e+08
  9.984989e+08]
 [9.994991e+08 9.994991e+08 9.994991e+08 ... 9.994991e+08 9.994991e+08
  9.994991e+08]]
[[4.9950000e+05 4.9950000e+05 4.9950000e+05 ... 4.9950000e+05
  4.9950000e+05 4.9950000e+05]
 [1.4995000e+06 1.4995000e+06 1.4995000e+06 ... 1.4995000e+06
  1.4995000e+06 1.4995000e+06]
 [2.4995000e+06 2.4995000e+06 2.4995000e+06 ... 2.4995000e+06
  2.4995000e+06 2.4995000e+06]
 ...
 [9.9749914e+08 9.9749914e+08 9.9749914e+08 ... 9.9749914e+08
  9.9749914e+08 9.9749914e+08]
 [9.9849901e+08 9.9849901e+08 9.9849901e+08 ... 9.9849901e+08
  9.9849901e+08 9.9849901e+08]
 [

In [ ]:
%%timeit
fast_matmul[blockspergrid, threadsperblock](x_d, y_d, z_d)
cuda.synchronize()

3.08 ms ± 47.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
